# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aminaljevakovich/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My chosen ML task is **ranking**.

The goal is to rank content pages by their estimated risk of declining performance, so the editorial team can prioritize which pages to review first.

Each page receives a score based on observable features such as content age, time since the last update, impressions, CTR, and average search position.

The output supports a practical content decision: which pages should be investigated first for possible updates.

Although predicting whether a page is declining can be treated as a classification problem, the main business objective is ranking because the team needs a prioritized list of pages rather than only a yes/no prediction.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

task_type = "Ranking"
unit_of_analysis = "One content page"
business_action = "Prioritize pages for editorial review"

print("ML task:", task_type)
print("Unit of analysis:", unit_of_analysis)
print("Business action:", business_action)


ML task: Ranking
Unit of analysis: One content page
Business action: Prioritize pages for editorial review


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is whether a content page is experiencing declining performance.

For this starter dataset, I will use a binary proxy label called `is_declining_label`:

- **1:** The page has `trend_direction = "down"`.
- **0:** The page does not have `trend_direction = "down"`.

This proxy is derived from the dataset's existing trend classification, rather than independently observed future outcomes.

The model would use available content and performance features to estimate the probability of decline. These probabilities could then be used to rank pages for editorial review.

I would exclude `trend_direction` and `trend_pct` from the model's input features because they directly reveal or determine the proxy label. For a production system, I would also need to verify that all input features were available before the outcome being predicted.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import os
import subprocess
import pandas as pd

repo_dir = "/content/flyrank-ml-internship-starter"
csv_path = os.path.join(
    repo_dir, "data/raw/content_refresh_anonymized.csv"
)

if not os.path.isfile(csv_path):
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
            repo_dir
        ],
        check=True
    )

df = pd.read_csv(csv_path)

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print(df["is_declining_label"].value_counts())
print("Declining rate:",
      round(df["is_declining_label"].mean(), 3))


is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is **Precision@50**.

Precision@50 measures the proportion of actually declining pages among the top 50 pages ranked by the model.

I chose this metric because an editorial team has limited time and needs to prioritize the most relevant pages for review.

As an initial benchmark, I will compare the model against a simple hand-written rule. In my previous exercise, the hand-written rule achieved Precision@50 of 0.680 on the full dataset.

My initial goal is to exceed this baseline on held-out data. A Precision@50 of 0.70 or higher would be a useful provisional target, but it is not yet a validated business threshold.

I would use client-holdout validation for a more realistic assessment of performance on unseen clients.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    top_indices = np.argsort(-scores)[:k]
    return labels[top_indices].mean()

k = 50
target_precision = 0.70

print("Primary metric: Precision@50")
print("Review capacity:", k, "pages")
print("Provisional target:", target_precision)
print("Target declining pages in top 50:",
      int(k * target_precision))


Primary metric: Precision@50
Review capacity: 50 pages
Provisional target: 0.7
Target declining pages in top 50: 35


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is **one content page**.

Each row in the dataset represents one page, with features describing its content and search performance.

Relevant features include content age, days since the last update, impressions over 90 days, average search position, CTR, and word count.

I use `is_declining_label` as the binary target proxy, where 1 indicates a declining trend and 0 indicates a non-declining trend.

The dataframe below shows real observations from the anonymized FlyRank starter dataset. The target is derived from `trend_direction` for this educational exercise, rather than independently measured future decline.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import subprocess
import pandas as pd

# Find the starter dataset in Colab or locally.
csv_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(csv_path):
    repo_dir = "/content/flyrank-ml-internship-starter"
    if not os.path.exists(repo_dir):
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                repo_dir
            ],
            check=True
        )
    csv_path = os.path.join(
        repo_dir, "data/raw/content_refresh_anonymized.csv"
    )

df = pd.read_csv(csv_path)

df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

display_columns = features + ["is_declining_label"]

print("Unit of analysis: one content page")
print("Dataset shape:", df.shape)
print("Declining rate:",
      round(df["is_declining_label"].mean(), 3))

display(df[display_columns].head(10))


Unit of analysis: one content page
Dataset shape: (30000, 45)
Declining rate: 0.542


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,1
1,445,25,15320,20.3,0.05,2481.0,1
2,141,20,12581,36.5,0.09,3515.0,1
3,463,22,11751,6.2,0.49,NaN,0
4,263,14,19140,44.0,0.13,2803.0,1
5,147,20,3970,8.5,0.03,3080.0,1
6,90,20,20,7.0,0.00,3059.0,1
7,445,22,1724,21.2,0.06,NaN,0
8,90,20,32574,46.0,0.09,3807.0,1
9,257,104,1240,4.9,0.16,NaN,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can prioritize pages based on simple conditions, such as pages that have not been updated for at least 180 days and have at least 500 impressions.

However, declining content performance may depend on several interacting factors, including content age, update frequency, search visibility, CTR, and average position.

A machine learning model can learn combinations of these signals instead of relying on manually chosen thresholds.

In my previous Week 2 experiment, the depth-3 decision tree achieved a test Precision@50 of 0.680, compared with 0.620 for the depth-2 tree on the same random train/test split.

This suggests that additional model complexity can help capture useful patterns, although the results do not prove that ML is always better than a fixed rule.

I would compare the model against a hand-written baseline on the same held-out data before recommending it for production use.

The final output would be a ranked list of pages for editorial review, helping the team decide which content to investigate first.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Example of a transparent hand-written ranking rule.

stale = (
    df["days_since_last_update"] >= 180
).astype(int)

visible = (
    df["impressions_90d"] >= 500
).astype(int)

df["hand_rule_score"] = (
    stale * visible * df["impressions_90d"]
)

baseline_precision = precision_at_k(
    df["hand_rule_score"],
    df["is_declining_label"],
    k=50
)

print("Hand-rule Precision@50:",
      round(baseline_precision, 3))

print("Top 10 pages selected by the hand rule:")

display(
    df.sort_values(
        "hand_rule_score",
        ascending=False
    )[
        [
            "days_since_last_update",
            "impressions_90d",
            "ctr",
            "is_declining_label",
            "hand_rule_score"
        ]
    ].head(10)
)


Hand-rule Precision@50: 0.68
Top 10 pages selected by the hand rule:


,days_since_last_update,impressions_90d,ctr,is_declining_label,hand_rule_score
16751,194,61678,0.15,1,61678
16514,194,59472,0.13,1,59472
7021,194,25715,0.23,1,25715
21268,193,13299,0.49,1,13299
11489,194,7812,0.01,1,7812
12045,193,7558,0.20,1,7558
698,194,4590,0.00,1,4590
5327,194,4556,0.33,1,4556
26810,194,4429,0.38,1,4429
20837,193,1697,0.12,1,1697


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.